In [ ]:
import os
import pandas as pd
import yfinance as yf

os.chdir(os.path.dirname(os.getcwd()))


from src.constants.config import path_portfolio, path_db
from src.constants.tickers import isin_to_ticker

In [ ]:
# load portfolio
df = pd.read_csv(path_portfolio)

df['Ticker'] = df['ISIN'].replace(isin_to_ticker)
df['SnapshotDate'] = pd.to_datetime(df['SnapshotDate']).dt.date

df_grouped = df.groupby(['Product', 'ISIN', 'Ticker']).agg({'SnapshotDate': ['min', 'max']})
df_grouped.columns = ['StartDate', 'EndDate']
df_grouped = df_grouped.reset_index()

In [ ]:

def collect_stock_rates(df):
    """Fetch historical stock data for a grouped DataFrame of tickers and return combined results."""
    all_data = []
    empty_data = []

    for _, row in df.iterrows():
        start_date = row['StartDate']
        end_date = row['EndDate']
        symbol = row['Ticker']
        product = row['Product']
        isin_code = row['ISIN']

        try:
            ticker = yf.Ticker(symbol)
            currency = ticker.info['currency']
            history_df = (
                ticker.history(start=start_date, end=end_date)
                .reset_index()[['Date', 'Close']]
                .assign(
                    Product=product,
                    ISIN=isin_code,
                    Ticker=symbol,
                    currency=currency
                )
            )
            print(f"Retrieved {history_df.shape[0]} rows for {symbol}")
            all_data.append(history_df)

        except Exception as error:
            print(f"Error fetching data for {symbol}: {error}")
            empty_data.append({
                "Ticker": symbol,
                "Product": product,
                "ISIN": isin_code,
                "StartDate": start_date,
                "EndDate": end_date,
                "Error": str(error)
            })

    combined_df = pd.concat(all_data, ignore_index=True)
    combined_df['Date'] = pd.to_datetime(combined_df['Date'].astype(str).str.slice(start=0,stop=10)).dt.date

    return combined_df, empty_data

def search_database(path_db):
    """Returns Delisted stock from manual history db"""
    csv_files = [
        pd.read_csv(os.path.join(path_db, file), sep=';')
        for file in os.listdir(path_db)
        if file.endswith('.csv')
    ]
    return pd.concat(csv_files, ignore_index=True)



In [ ]:
df_db = search_database(path_db)

df1, df2 = collect_stock_rates(df_grouped)

df = pd.concat([df_db,df1])


In [ ]:
# Collect currencies
currencies = ['EURUSD=X', 'GBPEUR=X']
currency_data = []

for c in currencies:
    print(f"Fetching currency data for: {c}")
    
    # Detect currency type
    if 'GBP' in c:
        currency = 'GBP'
    elif 'USD' in c:
        currency = 'USD'
    else:
        currency = 'Unknown'

    try:
        ticker = yf.Ticker(c)
        cur_df = (
            ticker.history(period='max')
            .reset_index()[['Date', 'Close']]
            .assign(Product=c, Currency=currency)
            .rename({'Close':'Fx_rate'},axis=1)
        )

        print(f"Retrieved {cur_df.shape[0]} rows for {c}")
        currency_data.append(cur_df)

    except Exception as e:
        print(f"Error fetching data for {c}: {e}")

combined_currency_df = pd.concat(currency_data, ignore_index=True)
combined_currency_df['Date'] = pd.to_datetime(combined_currency_df['Date']).dt.date